# Build MDE webapp data (`docs/data/mde.json`)

Exports the strong-perturbation MDE coords + cluster labels + n_DEGs as a compact JSON the static viewer (`docs/`) consumes.

**Inputs**
- `psp/notebooks/figs/_MDE_clusters.xlsx` — produced by `psp.da.compute_MDE_map` (x, y, gene_target, leiden cluster, hdbscan cluster) on the 1,655 strong perturbations.
- `psp/notebooks/input_files/n_degs_k562_kolf_rpe1.csv` — per-pert n_DEGs across cell types (we take the `KOLF` column).

**Output**
- `docs/data/mde.json` — list of `{g, x, y, l, h, n}` records.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/tscc/projects/ps-malilab/ydoctor/KOLF_Perturbation_Atlas')
MDE_XLSX   = ROOT / 'psp/notebooks/figs/_MDE_clusters.xlsx'
NDEGS_CSV  = ROOT / 'psp/notebooks/input_files/n_degs_k562_kolf_rpe1.csv'
OUT        = ROOT / 'docs/data/mde.json'

In [ ]:
mde = pd.read_excel(MDE_XLSX).rename(columns={
    'gene_target': 'gene',
    'leiden cluster': 'leiden',
    'hdbscan cluster': 'hdbscan',
})
mde.head()

In [ ]:
# n_DEGs lookup: the CSV is tab-separated; KOLF column has gene symbols
ndegs = pd.read_csv(NDEGS_CSV, sep='\t', index_col=0)
kolf_ndegs = dict(zip(
    ndegs['KOLF'].astype(str),
    pd.to_numeric(ndegs['Number of DEGs_KOLF'], errors='coerce')
))
mde['n_degs'] = mde['gene'].map(kolf_ndegs).fillna(-1).astype(int)
missing = int((mde['n_degs'] < 0).sum())
print(f'{len(mde)} perts; {missing} missing n_DEGs (set to -1)')

In [ ]:
records = [
    {
        'g': r.gene,
        'x': round(float(r.x), 3),
        'y': round(float(r.y), 3),
        'l': int(r.leiden),
        'h': int(r.hdbscan),
        'n': int(r.n_degs),
    }
    for r in mde.itertuples(index=False)
]

OUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUT, 'w') as f:
    json.dump(records, f, separators=(',', ':'))

print(f'wrote {OUT} ({OUT.stat().st_size/1024:.1f} KB, {len(records)} records)')

## Preview locally

```bash
cd docs && python -m http.server 8000
# then open http://localhost:8000
```

## Deploy on GitHub Pages

1. Commit `docs/` and push to `main`.
2. GitHub → repo **Settings → Pages**: set **Source = Deploy from a branch**, **Branch = `main` / folder `/docs`**. Save.
3. Site will be live at `https://y-doctor.github.io/KOLF2.1J_Perturbation_Cell_Atlas/`.